# 06 - Squad Optimisation
Using Integer Linear Programming (PuLP) to select the optimal 15-player FPL squad.
Constraints: £100m budget, positional limits, max 3 players per club, valid formation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pulp
import warnings
warnings.filterwarnings('ignore')

# Load predictions from both models
xgb_preds = pd.read_csv('../data/processed/xgb_predictions.csv')
nn_preds  = pd.read_csv('../data/processed/nn_predictions.csv')

# Load player metadata for price and team info
players_df = pd.read_csv('../data/processed/players_season.csv')
past_df    = pd.read_csv('../data/processed/player_past_seasons.csv')

print('Data loaded.')
print(xgb_preds.shape, nn_preds.shape)

Data loaded.
(289, 6) (289, 6)


In [2]:
# Build ensemble predictions (average of XGBoost and NN)
xgb_preds = xgb_preds.rename(columns={'predicted_points': 'xgb_pred'})
nn_preds  = nn_preds.rename(columns={'predicted_points': 'nn_pred'})

merged = xgb_preds[['name','position','actual_points','xgb_pred']].merge(
    nn_preds[['name','nn_pred']], on='name', how='inner'
)
merged['ensemble_pred'] = (merged['xgb_pred'] + merged['nn_pred']) / 2

# Add price and team from past seasons (2025/26)
latest = past_df[past_df['season_name'] == '2025/26'][['name','team','end_cost']].drop_duplicates('name')
latest['price'] = latest['end_cost'] / 10

merged = merged.merge(latest[['name','team','price']], on='name', how='left')
merged = merged.dropna(subset=['price','team'])

print(f'Players available for selection: {len(merged)}')
merged.head(10)

Players available for selection: 301


,name,position,actual_points,xgb_pred,nn_pred,ensemble_pred,team,price
0,Schade,MID,125,165.61823,137.48413,151.551180,Brentford,6.8
1,Raya,GKP,162,160.45294,135.11351,147.783225,Arsenal,6.2
2,Watkins,FWD,167,157.44272,139.72017,148.581445,Aston Villa,8.7
3,Semenyo,MID,202,154.11710,164.00299,159.060045,Man City,8.0
4,Mbeumo,MID,148,153.09296,145.44190,149.267430,Man Utd,8.3
5,Ndiaye,MID,128,150.98402,145.56114,148.272580,Everton,6.3
6,B.Fernandes,MID,235,149.22198,198.07008,173.646030,Man Utd,10.4
7,Amad,MID,91,149.21931,135.81374,142.516525,Man Utd,6.2
8,Haaland,FWD,239,148.17772,258.79218,203.484950,Man City,14.7
9,Gakpo,MID,131,147.33977,160.68639,154.013080,Liverpool,7.3


In [3]:
# ILP Squad Optimisation Function
def optimise_squad(players, budget=100.0, pred_col='ensemble_pred'):
    """
    Select optimal 15-player FPL squad using Integer Linear Programming.
    Constraints:
      - Total price <= budget
      - 2 GKP, 5 DEF, 5 MID, 3 FWD
      - Max 3 players per club
    """
    n = len(players)
    indices = list(range(n))

    prob = pulp.LpProblem('FPL_Squad_Optimisation', pulp.LpMaximize)

    # Decision variables
    x = [pulp.LpVariable(f'x_{i}', cat='Binary') for i in indices]

    # Objective: maximise predicted points
    prob += pulp.lpSum(players.iloc[i][pred_col] * x[i] for i in indices)

    # Budget constraint
    prob += pulp.lpSum(players.iloc[i]['price'] * x[i] for i in indices) <= budget

    # Squad size
    prob += pulp.lpSum(x[i] for i in indices) == 15

    # Positional constraints
    for pos, count in [('GKP', 2), ('DEF', 5), ('MID', 5), ('FWD', 3)]:
        prob += pulp.lpSum(x[i] for i in indices if players.iloc[i]['position'] == pos) == count

    # Max 3 per club
    for club in players['team'].unique():
        prob += pulp.lpSum(x[i] for i in indices if players.iloc[i]['team'] == club) <= 3

    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    selected = [i for i in indices if pulp.value(x[i]) == 1]
    squad = players.iloc[selected].copy()
    return squad

print('Optimiser function defined.')

Optimiser function defined.


In [4]:
# Run optimisation
squad = optimise_squad(merged, budget=100.0, pred_col='ensemble_pred')
squad = squad.sort_values('position')

total_price = squad['price'].sum()
total_actual = squad['actual_points'].sum()
total_predicted = squad['ensemble_pred'].sum()

print(f'Total squad price: £{total_price:.1f}m')
print(f'Total predicted points: {total_predicted:.1f}')
print(f'Total actual points: {total_actual:.1f}')
print()
squad[['name','position','team','price','ensemble_pred','actual_points']]

Total squad price: £100.0m
Total predicted points: 2102.9
Total actual points: 2142.0



,name,position,team,price,ensemble_pred,actual_points
13,J.Timber,DEF,Arsenal,6.0,133.982268,149
17,Virgil,DEF,Liverpool,6.1,133.359955,175
47,Gabriel,DEF,Arsenal,7.3,129.908103,209
48,Kerkez,DEF,Liverpool,5.6,115.858875,85
49,Murillo,DEF,Nott'm Forest,5.2,118.742030,83
2,Watkins,FWD,Aston Villa,8.7,148.581445,167
29,João Pedro,FWD,Chelsea,7.4,135.328015,177
35,Delap,FWD,Chelsea,6.2,135.172880,45
1,Raya,GKP,Arsenal,6.2,147.783225,162
16,Pickford,GKP,Everton,5.6,139.974450,135


In [5]:
# Visualise the optimal squad
fig, ax = plt.subplots(figsize=(12, 7))

pos_order = {'GKP': 0, 'DEF': 1, 'MID': 2, 'FWD': 3}
colors    = {'GKP': '#f1c40f', 'DEF': '#2ecc71', 'MID': '#3498db', 'FWD': '#e74c3c'}

for pos, grp in squad.groupby('position'):
    y = pos_order[pos]
    x_positions = np.linspace(0.1, 0.9, len(grp))
    for x_pos, (_, row) in zip(x_positions, grp.iterrows()):
        ax.add_patch(plt.Circle((x_pos, y), 0.04, color=colors[pos], zorder=3))
        ax.text(x_pos, y + 0.07, row['name'], ha='center', fontsize=7, fontweight='bold')
        ax.text(x_pos, y - 0.07, f"£{row['price']}m", ha='center', fontsize=6, color='gray')
        ax.text(x_pos, y - 0.12, f"{row['ensemble_pred']:.0f}pts", ha='center', fontsize=6, color='navy')

ax.set_xlim(0, 1)
ax.set_ylim(-0.5, 3.5)
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['GKP', 'DEF', 'MID', 'FWD'], fontsize=12)
ax.set_xticks([])
ax.set_facecolor('#2d5a27')
fig.patch.set_facecolor('#1a1a2e')
ax.tick_params(colors='white')
ax.set_title(f'Optimal FPL Squad | Budget: £{total_price:.1f}m | Predicted: {total_predicted:.0f}pts',
             color='white', fontsize=13, pad=15)

plt.tight_layout()
plt.savefig('../data/processed/optimal_squad.png', dpi=150, facecolor=fig.get_facecolor())
plt.show()

<Figure size 1200x700 with 1 Axes>

In [6]:
# Benchmark: compare vs average FPL manager
avg_fpl_points = 2150  # approximate average FPL season score

# Starting 11 points (top 11 by predicted from squad)
starting_11 = squad.nlargest(11, 'ensemble_pred')
starting_11_actual = starting_11['actual_points'].sum()

print(f'Our AI starting 11 actual points: {starting_11_actual}')
print(f'Average FPL manager season score: {avg_fpl_points}')
print(f'Difference: {starting_11_actual - avg_fpl_points:+.0f} points')

# Save squad
squad.to_csv('../data/processed/optimal_squad.csv', index=False)
print('\nOptimal squad saved.')

Our AI starting 11 actual points: 1590
Average FPL manager season score: 2150
Difference: -560 points

Optimal squad saved.
